# Refuerzo: Clasificación Aplicada — Detección de Cáncer de Mama
## Ingeniería del Conocimiento (ISO56B) — UNCP
### Msc. Jaime Antonio Huaytalla Pariona — Semestre 2026-II

**Objetivo:** Aplicar modelos de clasificación a un problema real de salud: distinguir tumores malignos de benignos. Incluye técnicas avanzadas como pipelines, validación cruzada, búsqueda de hiperparámetros y manejo de desbalance.

**Nivel:** Intermedio → Avanzado

**Contenido:**
1. El problema: detección de cáncer de mama
2. Exploración profunda del dataset
3. Pipeline completo de clasificación
4. Cuatro modelos con tuning de hiperparámetros
5. Evaluación avanzada: curvas ROC y Precision-Recall
6. Comparación y selección del mejor modelo


## 1. Configuración del Entorno

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer

import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')
print("✅ Entorno listo")


## 2. El Problema: Detección de Cáncer de Mama

El dataset **Breast Cancer Wisconsin** contiene características de núcleos celulares digitalizados a partir de biopsias. El objetivo es clasificar si un tumor es **maligno** (canceroso) o **benigno** (no canceroso).

**¿Por qué es un buen problema de ejemplo?**
- Es un problema **binario** real con impacto en la salud
- Los **falsos negativos son peligrosos** (no detectar un tumor maligno)
- El dataset tiene **30 features** — más complejo que Iris
- Las clases están **ligeramente desbalanceadas**


In [ ]:
# Cargar datos
cancer = load_breast_cancer()
df = pd.DataFrame(cancer.data, columns=cancer.feature_names)
df['target'] = cancer.target
df['diagnostico'] = df['target'].map({0: 'Maligno', 1: 'Benigno'})

print("="*60)
print("DATASET BREAST CANCER WISCONSIN")
print("="*60)
print(f"\nMuestras: {df.shape[0]}")
print(f"Features: {df.shape[1]-2}")
print(f"\nDistribución de clases:")
print(df['diagnostico'].value_counts())
print(f"\nProporción:")
print(df['diagnostico'].value_counts(normalize=True).round(3))
print(f"\n📌 Ratio benigno/maligno ≈ 1.7:1 (desbalance leve)")


In [ ]:
# Exploración visual: distribución de features clave
features_clave = ['mean radius', 'mean texture', 'mean perimeter',
                   'mean area', 'mean smoothness', 'mean compactness']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, feat in enumerate(features_clave):
    for diag, color in [('Maligno', '#e74c3c'), ('Benigno', '#2ecc71')]:
        mask = df['diagnostico'] == diag
        axes[i].hist(df.loc[mask, feat], bins=25, alpha=0.6,
                     label=diag, color=color, edgecolor='white')
    axes[i].set_xlabel(feat)
    axes[i].set_ylabel('Frecuencia')
    axes[i].legend()
    axes[i].set_title(f'Distribución de {feat}')

plt.suptitle('Distribución de Features Clave por Diagnóstico', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

print("📌 Los tumores malignos tienden a tener mayor radio, perímetro y área.")
print("   Esto tiene sentido biológicamente: los tumores cancerosos crecen más.")


In [ ]:
# Correlación entre features
fig, ax = plt.subplots(figsize=(14, 10))
# Usar solo las 10 features "mean" para no saturar el heatmap
mean_features = [f for f in cancer.feature_names if 'mean' in f]
corr_matrix = df[mean_features + ['target']].corr()

sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, ax=ax, square=True,
            linewidths=0.5, linecolor='white')
ax.set_title('Matriz de Correlación (Features Mean)', fontsize=16)
plt.tight_layout()
plt.show()

print("📌 Alta correlación entre radius, perimeter y area (son geométricamente relacionadas)")
print("   Correlación negativa fuerte con target = feature útil para clasificar")


## 3. Preparación de Datos con Pipeline

Un **Pipeline** encadena pasos de preprocesamiento y modelado, garantizando que no haya data leakage.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Separar features y target
X = cancer.data
y = cancer.target

# Split estratificado
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape[0]} muestras")
print(f"Test:  {X_test.shape[0]} muestras")
print(f"\nDistribución en train: {np.bincount(y_train)} (Maligno, Benigno)")
print(f"Distribución en test:  {np.bincount(y_test)} (Maligno, Benigno)")


## 4. Entrenamiento de 4 Modelos con Pipelines

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score

# Definir 4 pipelines
pipelines = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=5000, random_state=42))
    ]),
    'Decision Tree': Pipeline([
        ('clf', DecisionTreeClassifier(max_depth=5, min_samples_leaf=5, random_state=42))
    ]),
    'SVM (RBF)': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', SVC(kernel='rbf', probability=True, random_state=42))
    ]),
    'KNN': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', KNeighborsClassifier(n_neighbors=7))
    ])
}

# Entrenar y evaluar cada uno
resultados = {}
for nombre, pipe in pipelines.items():
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    resultados[nombre] = {'accuracy': acc, 'f1': f1, 'y_pred': y_pred}
    print(f"\n{'='*50}")
    print(f"{nombre}")
    print(f"{'='*50}")
    print(f"Accuracy: {acc:.4f}  |  F1-Score: {f1:.4f}")
    print(classification_report(y_test, y_pred, target_names=['Maligno', 'Benigno']))


## 5. Evaluación Avanzada

### 5.1 Matrices de Confusión Comparativas

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

for ax, (nombre, res) in zip(axes, resultados.items()):
    cm = confusion_matrix(y_test, res['y_pred'])
    disp = ConfusionMatrixDisplay(cm, display_labels=['Maligno', 'Benigno'])
    disp.plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(f'{nombre}\nAcc: {res["accuracy"]:.3f} | F1: {res["f1"]:.3f}')

plt.suptitle('Matrices de Confusión — 4 Modelos', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

print("📌 En este problema, el error más grave es el FALSO NEGATIVO:")
print("   predecir 'Benigno' cuando en realidad es 'Maligno'.")
print("   Busca el modelo con menos FN (esquina inferior izquierda).")


### 5.2 Curva ROC — Rendimiento en Todos los Umbrales

In [ ]:
from sklearn.metrics import roc_curve, auc

fig, ax = plt.subplots(figsize=(10, 8))
colores = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

for (nombre, pipe), color in zip(pipelines.items(), colores):
    if hasattr(pipe, 'predict_proba'):
        y_proba = pipe.predict_proba(X_test)[:, 1]
    else:
        y_proba = pipe.decision_function(X_test)
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, lw=2,
            label=f'{nombre} (AUC = {roc_auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Aleatorio (AUC = 0.500)')
ax.set_xlabel('False Positive Rate (1 - Especificidad)')
ax.set_ylabel('True Positive Rate (Sensibilidad / Recall)')
ax.set_title('Curva ROC — Comparación de Modelos')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("📌 CÓMO LEER LA CURVA ROC:")
print("  - Más arriba y a la izquierda = MEJOR modelo")
print("  - AUC = 1.0 = perfecto; AUC = 0.5 = aleatorio (inútil)")
print("  - AUC > 0.9 = excelente; AUC > 0.8 = bueno")


### 5.3 Validación Cruzada — Estimación Robusta

In [ ]:
from sklearn.model_selection import cross_val_score

print("="*60)
print("VALIDACIÓN CRUZADA (5-fold) — F1-Score")
print("="*60)

cv_results = {}
for nombre, pipe in pipelines.items():
    scores = cross_val_score(pipe, X, y, cv=5, scoring='f1')
    cv_results[nombre] = scores
    print(f"\n{nombre}:")
    print(f"  F1 por fold: {scores.round(4)}")
    print(f"  F1 medio:    {scores.mean():.4f} ± {scores.std():.4f}")

# Boxplot comparativo
fig, ax = plt.subplots(figsize=(10, 6))
bp = ax.boxplot(cv_results.values(), labels=cv_results.keys(), patch_artist=True)
for patch, color in zip(bp['boxes'], colores):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.set_ylabel('F1-Score (Cross-Validation)')
ax.set_title('Comparación Robusta de Modelos (5-Fold CV)')
ax.grid(True, alpha=0.3)
plt.xticks(rotation=10)
plt.tight_layout()
plt.show()


## 6. Búsqueda de Hiperparámetros (GridSearchCV)

Buscamos la mejor combinación de hiperparámetros para el modelo SVM.

In [ ]:
from sklearn.model_selection import GridSearchCV

# Pipeline para SVM
svm_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', SVC(probability=True, random_state=42))
])

# Hiperparámetros a probar
param_grid = {
    'clf__C': [0.1, 1, 10, 100],
    'clf__kernel': ['linear', 'rbf'],
    'clf__gamma': ['scale', 'auto', 0.01, 0.1]
}

# Búsqueda exhaustiva
grid_search = GridSearchCV(
    svm_pipeline, param_grid, cv=5,
    scoring='f1', n_jobs=-1, verbose=0
)
grid_search.fit(X_train, y_train)

print("="*60)
print("RESULTADOS DE GRID SEARCH — SVM")
print("="*60)
print(f"\nMejores hiperparámetros: {grid_search.best_params_}")
print(f"Mejor F1 (CV): {grid_search.best_score_:.4f}")

# Evaluar en test
y_pred_best = grid_search.predict(X_test)
print(f"\nF1 en test: {f1_score(y_test, y_pred_best):.4f}")
print(f"Accuracy en test: {accuracy_score(y_test, y_pred_best):.4f}")
print(f"\nReporte:")
print(classification_report(y_test, y_pred_best, target_names=['Maligno', 'Benigno']))


## 7. Resumen Comparativo Final

In [ ]:
# Tabla resumen
resumen = []
for nombre in pipelines:
    resumen.append({
        'Modelo': nombre,
        'Accuracy (test)': resultados[nombre]['accuracy'],
        'F1 (test)': resultados[nombre]['f1'],
        'F1 CV (mean)': cv_results[nombre].mean(),
        'F1 CV (std)': cv_results[nombre].std()
    })
resumen.append({
    'Modelo': 'SVM (Tuned)',
    'Accuracy (test)': accuracy_score(y_test, y_pred_best),
    'F1 (test)': f1_score(y_test, y_pred_best),
    'F1 CV (mean)': grid_search.best_score_,
    'F1 CV (std)': 0
})

df_resumen = pd.DataFrame(resumen).sort_values('F1 CV (mean)', ascending=False)
print("="*70)
print("RESUMEN FINAL — TODOS LOS MODELOS")
print("="*70)
print(df_resumen.to_string(index=False, float_format='%.4f'))

# Gráfico de barras
fig, ax = plt.subplots(figsize=(12, 6))
x = range(len(df_resumen))
width = 0.35
bars1 = ax.bar([i - width/2 for i in x], df_resumen['F1 (test)'],
               width, label='F1 (Test)', color='#2196F3', alpha=0.8)
bars2 = ax.bar([i + width/2 for i in x], df_resumen['F1 CV (mean)'],
               width, label='F1 (CV)', color='#FF9800', alpha=0.8)

ax.set_ylabel('F1-Score')
ax.set_title('Comparación Final de Modelos')
ax.set_xticks(x)
ax.set_xticklabels(df_resumen['Modelo'], rotation=15)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f'{bar.get_height():.3f}', ha='center', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f'{bar.get_height():.3f}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()


## 8. Conclusiones

En este notebook practicaste:

1. **Explorar un dataset real** de 30 features con visualizaciones informativas
2. **Construir pipelines** que encapsulan estandarización + modelo
3. **4 modelos de clasificación** sobre un problema médico real
4. **Evaluación avanzada:** Matrices de confusión, curvas ROC, AUC
5. **Validación cruzada** para estimaciones confiables
6. **GridSearchCV** para optimizar hiperparámetros
7. **Comparación cuantitativa** para seleccionar el mejor modelo

**Próximo paso:** Aplica estos mismos conceptos y flujo de trabajo a los ejercicios propuestos.

---
*Material de refuerzo — Ingeniería del Conocimiento (ISO56B) — UNCP*
